# 02. Neural Network classification with PyTorch


In [ ]:
## 1. Make Classification Data and get it ready

import sklearn
from sklearn.datasets import make_circles


In [ ]:
# Make 1000 samples
n_samples = 1000

# Create circles

X, y = make_circles(n_samples,
                    noise = 0.03,
                    random_state=42)

In [ ]:
len(X), len(y)

In [ ]:
print(X[:5])

In [ ]:
print(y[:5])

In [ ]:
# Make DataFrame of circle data
import pandas as pd
circles = pd.DataFrame({"X1": X[:,0], 
                        "X2": X[:, 1],
                        "label": y})
circles.head()

In [ ]:
# Visualize, visualize, visualize
import matplotlib.pyplot as plt
plt.scatter(x=X[:, 0],
            y = X[:,1],
            c=y,
            cmap = plt.cm.RdYlBu);

In [ ]:
# Note: The data we're working with is often referred to as a toy dataset

In [ ]:
### 1.1 Check input and output shapes
X.shape, y.shape

In [ ]:
# View the first example of features and labels
X_sample = X[0]
y_sample = y[0]

print(f"Values for one sample of X: {X_sample} and the same for y: {y_sample}")
print(f"Shapes for one sample of X: {X_sample.shape} and the same for y: {y_sample.shape}")

In [ ]:
#### 1.2 Turn data into tensors and create train and test splits
import torch
torch.__version__

In [ ]:
X = torch.from_numpy(X).type(torch.float32)


In [ ]:
y = torch.from_numpy(y).type(torch.float32)

In [ ]:
type(X), X.dtype, y.dtype

In [ ]:
#Split data into training and test sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size=0.2,
                                                    random_state=42)

In [ ]:
len(X_train), len(X_test), len(y_train), len(y_test)

In [ ]:
## 2. Building a model

# Classifying a our blue and red dots
# Setup device agnostic code -> construct a model -> define loss function and optimizer -> create training data

In [ ]:
from torch import nn

# Made device agnostic code
device =  "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
# Model creation Subclass nn.Module -> Create 2 nn.linear layers that are capable of handling the shapes of our data -> forward -> instatiate instance and send to target device


In [ ]:
# 1. Construct Model Class
class CircleModelV0(nn.Module):
    def __init__(self):
        super().__init__()
        self.two_linear_layers =  nn.Sequential(nn.Linear(in_features=2, out_features=5),
                                                nn.Linear(in_features=5, out_features=1))
    def forward(self, x):
        return self.two_linear_layers(x)

model_0 = CircleModelV0().to(device)
model_0


In [ ]:
next(model_0.parameters()).device

In [ ]:
# Make predictions
untrained_preds = model_0(X_test)
print(f"Length of predictions: {len(untrained_preds)}, Shape: {untrained_preds.shape}")
print(f"Length of test samples: {len(X_test)}, Shape: {X_test.shape}")
print(f"\nFirst 10 predictions: \n{untrained_preds[:10]}")
print(f"\nFirst 10 labels:\n{y_test[:10]}")


In [ ]:
untrained_preds

In [ ]:
model_0.state_dict()

In [ ]:
# Make predictions
with torch.inference_mode():
    untrained_preds = model_0(X_test.to(device))
print(f"Length of predictions: {len(untrained_preds)}, Shape: {untrained_preds.shape}")
print(f"Length of test samples: {len(X_test)}, Shape: {X_test.shape}")
print(f"\nFirst 10 predictions: \n{untrained_preds[:10]}")
print(f"\nFirst 10 labels:\n{y_test[:10]}")

In [ ]:
X_test[:10], y_test[:10]

In [ ]:
### 2.1 Setup loss function and optimizer

loss_fn = nn.BCEWithLogitsLoss() # BCEL

optimizer = torch.optim.SGD(params= model_0.parameters(),
                            lr= 0.1)

In [ ]:
model_0.state_dict()

In [ ]:
# Calculate accuracy -  out of 100 examples, what percentage does our model get right?

def accuracy_fn(y_true, y_pred):
    correct = torch.eq(y_true, y_pred).sum().item()
    acc = (correct/len(y_pred))*100
    return acc

In [ ]:
with torch.inference_mode():
    y_logits = model_0(X_test.to(device))[:5]
y_logits

In [ ]:
## 3.Train model
y_pred_probs = torch.sigmoid(y_logits)
y_pred_probs

In [ ]:
# Use the sigmoid activiation function on our model logits to turn them ino prediction probabilities
y_preds = torch.round(y_pred_probs)

y_preds_labels = torch.round(torch.sigmoid(model_0(X_test.to(device))[:5]))

print(torch.eq(y_preds.squeeze(), y_preds_labels.squeeze()))

y_preds.squeeze()

In [ ]:
y_test[:5]

In [ ]:
X_test, y_test

In [ ]:
### 3.2 Build a train and test loop
torch.manual_seed(42)

EPOCH = 100

X_train, y_train = X_train.to(device), y_train.to(device)

X_test, y_test = X_test.to(device), y_test.to(device)
for epoch in range(EPOCH):

        # Training
        model_0.train()

        # 1. Forward Pass
        y_logits = model_0(X_train).squeeze()
        y_pred = torch.round(torch.sigmoid(y_logits))

        # 2. Calculate Loss    
        loss = loss_fn(y_logits, # nn.BCEWithLogitsLoss expects raw logits as input 
                    y_train)
        acc = accuracy_fn(y_true = y_train,
                        y_pred=y_pred)

        #3 Optimizer zero grad

        optimizer.zero_grad()

        # 4. Loss backward
        loss.backward()

        # 5. Optimizer step (gradient descent)
        optimizer.step()

        # Testing
        model_0.eval()

        if epoch % 5:
            with torch.inference_mode():
                # 1. Forward Pass
                test_logits = model_0(X_test).squeeze()
                test_pred = torch.round(torch.sigmoid(test_logits))

                # 2. Calculate test loss/acc
                test_loss = loss_fn(test_logits,
                                    y_test)
                test_acc = accuracy_fn(y_true=y_test,
                                    y_pred=test_pred)
                print(f"Epoch:{epoch} | Loss:{loss:.5f} | Accuracy:{acc:.2f}% | Test Loss:{test_loss:.5f} | Test acc:{test_acc:.2f}%")

In [ ]:
## 4. Make predictions and evaluate the model -> what's going on?
import requests

from pathlib import Path

# Download helper functions from PyTorch repo (if it's not already downloaded)
if Path("helper_functions.py").is_file():
    print("helper_functions.py already exists, skipping download")
else:
    print("Download helper_functions.py")
    request = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/helper_functions.py")
    with open("helper_functions.py", "wb") as f:
        f.write(request.content)

from helper_functions import plot_predictions, plot_decision_boundary

In [ ]:
# Plot decision boundary of the model
plt.figure(figsize=(12,6))
plt.subplot(1,2,1)
plt.title("Train")
plot_decision_boundary(model_0,X_train,y_train)
plt.subplot(1,2,2)
plt.title("Test")
plot_decision_boundary(model_0,X_test, y_test)

# Improving a model

1. Add more layers - give the model more chances to learn about patterns in the data
2. Add more hidden layers - go from 5 hidden units to 10 hidden units
3. Fit for longer - give the model more chance to learn
4. Changing the activation functions 
5. Change the learning rate 
6. Change the loss function

These options are from the model perspective, can also improve from a data perspective

In [ ]:
class CircleModelv1(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(in_features = 2,
                                 out_features = 10)
        self.layer_2 = nn.Linear(in_features = 10,
                                 out_features=10)
        self.layer_3 = nn.Linear(in_features=10, out_features=1)
    def forward(self, x):
        return self.layer_3(self.layer_2(self.layer_1(x))) # this way leverages speed ups where possible behind the scenes
model_1 = CircleModelv1().to(device)
model_1

In [ ]:
# create a loss function
loss_fn = nn.BCEWithLogitsLoss()
#Create optimizer
optimizer = torch.optim.SGD(params=model_1.parameters(),
                            lr = 0.1)
#Write a training loop

epoch = 1000
torch.manual_seed(42)
for e in range(epoch):

    model_1.train()

    # Step 1: Forward Pass
    y_logits = model_1(X_train).squeeze()
    y_pred = torch.round(torch.sigmoid(y_logits)) 

    # Step 2: Calculate loss
    loss = loss_fn(y_logits,
                   y_train)
    acc = accuracy_fn(y_train, y_pred)

    # Step 3: zero grad
    optimizer.zero_grad()

    # Step 4: Backpropogation
    loss.backward()

    # Step 5: Step
    optimizer.step()

    model_1.eval()

    #Check results

    if e % 100 == 0:
        with torch.inference_mode():

            #Forward pass for test data
            y_logits_test = model_1(X_test).squeeze()
            y_pred_test = torch.round(torch.sigmoid(y_logits_test)) 

            # Test loss
            test_loss = loss_fn(y_logits_test,y_test)
            test_acc = accuracy_fn(y_test,y_pred_test)
            print(f"Epoch:{epoch} | Loss:{loss:.5f} | Accuracy:{acc:.2f}% | Test Loss:{test_loss:.5f} | Test acc:{test_acc:.2f}%")

            

In [ ]:
# Plot decision boundary of the model
plt.figure(figsize=(12,6))
plt.subplot(1,2,1)
plt.title("Train")
plot_decision_boundary(model_1,X_train,y_train)
plt.subplot(1,2,2)
plt.title("Test")
plot_decision_boundary(model_1,X_test, y_test)

In [ ]:
# Can the model fit a single line
# Create some data (same as notebook 01)
weight = 0.7
bias = 0.3
start = 0
end = 1
step = 0.01

#Create data
X_regression = torch.arange()